In [1]:
"""
Notebook 6: Helper Functions and Additional Analysis Tools
Deteksi Anomali Transaksi Real-Time Menggunakan LSTM-Autoencoder
"""

# ============================================================================
# CELL 1: Import Libraries
# ============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:
# ============================================================================
# CELL 2: Model Loading Helper Functions
# ============================================================================
print("\n" + "="*80)
print("DEFINING MODEL LOADING HELPERS")
print("="*80)

class LSTMAutoencoder(nn.Module):
    """LSTM-Autoencoder for sequence anomaly detection"""
    def __init__(self, n_features, timestep, hidden_dim1=64, hidden_dim2=32, 
                 latent_dim=16, dropout=0.2):
        super(LSTMAutoencoder, self).__init__()
        
        self.n_features = n_features
        self.timestep = timestep
        self.hidden_dim1 = hidden_dim1
        self.hidden_dim2 = hidden_dim2
        self.latent_dim = latent_dim
        
        self.encoder_lstm1 = nn.LSTM(n_features, hidden_dim1, 1, batch_first=True, dropout=0)
        self.encoder_lstm2 = nn.LSTM(hidden_dim1, hidden_dim2, 1, batch_first=True, dropout=0)
        self.encoder_dense = nn.Linear(hidden_dim2, latent_dim)
        self.dropout = nn.Dropout(dropout)
        
        self.decoder_dense = nn.Linear(latent_dim, hidden_dim2)
        self.decoder_lstm1 = nn.LSTM(hidden_dim2, hidden_dim2, 1, batch_first=True, dropout=0)
        self.decoder_lstm2 = nn.LSTM(hidden_dim2, hidden_dim1, 1, batch_first=True, dropout=0)
        self.output_layer = nn.Linear(hidden_dim1, n_features)
        
    def forward(self, x):
        enc1, _ = self.encoder_lstm1(x)
        enc2, _ = self.encoder_lstm2(enc1)
        encoded = enc2[:, -1, :]
        latent = torch.relu(self.encoder_dense(encoded))
        latent = self.dropout(latent)
        
        decoded = torch.relu(self.decoder_dense(latent))
        decoded = decoded.unsqueeze(1).repeat(1, self.timestep, 1)
        
        dec1, _ = self.decoder_lstm1(decoded)
        dec2, _ = self.decoder_lstm2(dec1)
        output = self.output_layer(dec2)
        
        return output

def load_lstm_model(model_path, n_features, timestep, device='cuda'):
    """Load trained LSTM-Autoencoder model"""
    model = LSTMAutoencoder(n_features, timestep).to(device)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return model, checkpoint

print("✓ Model loading helpers defined")



DEFINING MODEL LOADING HELPERS
✓ Model loading helpers defined


In [3]:
# ============================================================================
# CELL 3: Prediction Helper Functions
# ============================================================================
print("\n" + "="*80)
print("DEFINING PREDICTION HELPERS")
print("="*80)

def predict_anomalies(model, X_data, threshold, device='cuda', batch_size=256):
    """
    Make predictions using LSTM-Autoencoder
    
    Returns:
        predictions: binary predictions (0=normal, 1=fraud)
        errors: reconstruction errors
        reconstructions: reconstructed sequences
    """
    test_dataset = TensorDataset(torch.FloatTensor(X_data))
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    errors = []
    reconstructions = []
    
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            inputs = batch[0].to(device)
            outputs = model(inputs)
            
            # Calculate reconstruction error
            batch_errors = torch.mean((inputs - outputs) ** 2, dim=(1, 2))
            errors.extend(batch_errors.cpu().numpy())
            reconstructions.append(outputs.cpu().numpy())
    
    errors = np.array(errors)
    reconstructions = np.vstack(reconstructions)
    predictions = (errors > threshold).astype(int)
    
    return predictions, errors, reconstructions

def batch_predict(model, X_data, threshold, device='cuda', chunk_size=10000):
    """
    Predict on large datasets in chunks to avoid memory issues
    """
    n_samples = len(X_data)
    all_predictions = []
    all_errors = []
    
    for i in range(0, n_samples, chunk_size):
        end_idx = min(i + chunk_size, n_samples)
        X_chunk = X_data[i:end_idx]
        
        predictions, errors, _ = predict_anomalies(model, X_chunk, threshold, device)
        all_predictions.append(predictions)
        all_errors.append(errors)
        
        print(f"Processed {end_idx}/{n_samples} samples...")
    
    return np.concatenate(all_predictions), np.concatenate(all_errors)

print("✓ Prediction helpers defined")


DEFINING PREDICTION HELPERS
✓ Prediction helpers defined


In [4]:
# ============================================================================
# CELL 4: Visualization Helper Functions
# ============================================================================
print("\n" + "="*80)
print("DEFINING VISUALIZATION HELPERS")
print("="*80)

def plot_sequence_reconstruction(original, reconstructed, idx=0, feature_names=None):
    """
    Plot original vs reconstructed sequence for a single sample
    """
    timesteps = original.shape[1]
    n_features = original.shape[2]
    
    # Select first few features to visualize
    n_features_to_plot = min(4, n_features)
    
    fig, axes = plt.subplots(n_features_to_plot, 1, figsize=(12, 3*n_features_to_plot))
    
    if n_features_to_plot == 1:
        axes = [axes]
    
    for i in range(n_features_to_plot):
        axes[i].plot(range(timesteps), original[idx, :, i], 
                    label='Original', marker='o', linewidth=2)
        axes[i].plot(range(timesteps), reconstructed[idx, :, i], 
                    label='Reconstructed', marker='x', linewidth=2, linestyle='--')
        
        feature_name = feature_names[i] if feature_names else f'Feature {i}'
        axes[i].set_ylabel(feature_name)
        axes[i].set_xlabel('Timestep')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

def plot_error_analysis(errors, labels, threshold, title='Error Analysis'):
    """
    Plot error distribution with threshold
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Separate errors by class
    normal_errors = errors[labels == 0]
    fraud_errors = errors[labels == 1]
    
    # Histogram
    axes[0].hist(normal_errors, bins=50, alpha=0.6, label='Normal', 
                color='green', edgecolor='black', density=True)
    axes[0].hist(fraud_errors, bins=50, alpha=0.6, label='Fraud', 
                color='red', edgecolor='black', density=True)
    axes[0].axvline(threshold, color='blue', linestyle='--', linewidth=2, 
                   label=f'Threshold: {threshold:.4f}')
    axes[0].set_xlabel('Reconstruction Error')
    axes[0].set_ylabel('Density')
    axes[0].set_title(f'{title} - Distribution')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Box plot
    axes[1].boxplot([normal_errors, fraud_errors], 
                   labels=['Normal', 'Fraud'])
    axes[1].axhline(threshold, color='blue', linestyle='--', linewidth=2,
                   label=f'Threshold: {threshold:.4f}')
    axes[1].set_ylabel('Reconstruction Error')
    axes[1].set_title(f'{title} - Box Plot')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    return fig

def plot_confusion_matrix_detailed(y_true, y_pred, title='Confusion Matrix'):
    """
    Plot detailed confusion matrix with percentages
    """
    from sklearn.metrics import confusion_matrix
    
    cm = confusion_matrix(y_true, y_pred)
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    
    # Create annotations with both counts and percentages
    annot = np.empty_like(cm, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f'{cm[i, j]}\n({cm_percent[i, j]:.1f}%)'
    
    sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax,
                xticklabels=['Normal', 'Fraud'],
                yticklabels=['Normal', 'Fraud'],
                cbar_kws={'label': 'Count'})
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)
    
    plt.tight_layout()
    return fig

print("✓ Visualization helpers defined")


DEFINING VISUALIZATION HELPERS
✓ Visualization helpers defined


In [5]:
# ============================================================================
# CELL 5: Metrics Calculation Helpers
# ============================================================================
print("\n" + "="*80)
print("DEFINING METRICS HELPERS")
print("="*80)

def calculate_all_metrics(y_true, y_pred, y_scores=None):
    """
    Calculate comprehensive metrics for anomaly detection
    """
    from sklearn.metrics import (
        confusion_matrix, accuracy_score, precision_score, 
        recall_score, f1_score, roc_auc_score, average_precision_score
    )
    
    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    # Basic metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    # Additional metrics
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    
    metrics = {
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Specificity': specificity,
        'FPR': fpr,
        'FNR': fnr
    }
    
    # Add AUC metrics if scores provided
    if y_scores is not None and len(np.unique(y_true)) > 1:
        try:
            metrics['ROC-AUC'] = roc_auc_score(y_true, y_scores)
            metrics['PR-AUC'] = average_precision_score(y_true, y_scores)
        except:
            metrics['ROC-AUC'] = None
            metrics['PR-AUC'] = None
    
    return metrics

def compare_models(results_dict, metric='F1-Score'):
    """
    Compare multiple models on a specific metric
    
    Args:
        results_dict: dict of {model_name: metrics_dict}
        metric: metric to compare
    """
    models = list(results_dict.keys())
    values = [results_dict[model][metric] for model in models]
    
    # Create comparison plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    
    bars = ax.bar(models, values, alpha=0.8, edgecolor='black')
    
    # Color bars
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(models)))
    for bar, color in zip(bars, colors):
        bar.set_color(color)
    
    # Add value labels on bars
    for i, (model, value) in enumerate(zip(models, values)):
        ax.text(i, value, f'{value:.4f}', 
               ha='center', va='bottom', fontweight='bold')
    
    ax.set_ylabel(metric, fontsize=12)
    ax.set_title(f'Model Comparison - {metric}', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, max(values) * 1.15])
    
    plt.xticks(rotation=15, ha='right')
    plt.tight_layout()
    
    return fig

print("✓ Metrics helpers defined")


DEFINING METRICS HELPERS
✓ Metrics helpers defined


In [6]:
# ============================================================================
# CELL 6: Threshold Optimization Helper
# ============================================================================
print("\n" + "="*80)
print("DEFINING THRESHOLD OPTIMIZATION HELPER")
print("="*80)

def find_optimal_threshold(errors, labels, metric='f1', percentile_range=None):
    """
    Find optimal threshold by maximizing specified metric
    
    Args:
        errors: reconstruction errors
        labels: true labels
        metric: 'f1', 'precision', 'recall', or 'accuracy'
        percentile_range: tuple (min, max) percentiles to test
    
    Returns:
        optimal_threshold, best_metric_value, threshold_curve
    """
    from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
    
    # Get normal errors for percentile calculation
    normal_errors = errors[labels == 0]
    
    # Define percentile range
    if percentile_range is None:
        percentiles = np.linspace(90, 99.9, 100)
    else:
        percentiles = np.linspace(percentile_range[0], percentile_range[1], 100)
    
    results = []
    
    for p in percentiles:
        threshold = np.percentile(normal_errors, p)
        predictions = (errors > threshold).astype(int)
        
        if metric == 'f1':
            score = f1_score(labels, predictions, zero_division=0)
        elif metric == 'precision':
            score = precision_score(labels, predictions, zero_division=0)
        elif metric == 'recall':
            score = recall_score(labels, predictions, zero_division=0)
        elif metric == 'accuracy':
            score = accuracy_score(labels, predictions)
        else:
            raise ValueError(f"Unknown metric: {metric}")
        
        results.append({
            'percentile': p,
            'threshold': threshold,
            'score': score
        })
    
    results_df = pd.DataFrame(results)
    best_idx = results_df['score'].idxmax()
    optimal_threshold = results_df.loc[best_idx, 'threshold']
    best_score = results_df.loc[best_idx, 'score']
    
    # Plot threshold curve
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    ax.plot(results_df['percentile'], results_df['score'], linewidth=2)
    ax.axvline(results_df.loc[best_idx, 'percentile'], 
              color='r', linestyle='--', linewidth=2,
              label=f'Optimal: P{results_df.loc[best_idx, "percentile"]:.1f}')
    ax.set_xlabel('Percentile', fontsize=12)
    ax.set_ylabel(metric.capitalize(), fontsize=12)
    ax.set_title(f'Threshold Optimization - {metric.capitalize()}', 
                fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    
    return optimal_threshold, best_score, results_df, fig

print("✓ Threshold optimization helper defined")


DEFINING THRESHOLD OPTIMIZATION HELPER
✓ Threshold optimization helper defined


In [7]:
# ============================================================================
# CELL 7: Data Export Helpers
# ============================================================================
print("\n" + "="*80)
print("DEFINING DATA EXPORT HELPERS")
print("="*80)

def export_results_to_latex(results_df, filename='table.tex', caption='Results', label='tab:results'):
    """
    Export pandas DataFrame to LaTeX table format
    """
    latex_str = results_df.to_latex(
        index=False,
        float_format="%.4f",
        caption=caption,
        label=label,
        position='htbp'
    )
    
    with open(filename, 'w') as f:
        f.write(latex_str)
    
    print(f"✓ LaTeX table exported to {filename}")
    return latex_str

def create_results_summary(metrics_dict, dataset_name='Dataset'):
    """
    Create a formatted summary of results
    """
    summary = f"""
{'='*60}
RESULTS SUMMARY - {dataset_name}
{'='*60}

Performance Metrics:
  • Accuracy:    {metrics_dict['Accuracy']:.4f}
  • Precision:   {metrics_dict['Precision']:.4f}
  • Recall:      {metrics_dict['Recall']:.4f}
  • F1-Score:    {metrics_dict['F1-Score']:.4f}
  • Specificity: {metrics_dict['Specificity']:.4f}

Confusion Matrix:
  • True Positives (TP):   {metrics_dict['TP']:,}
  • False Positives (FP):  {metrics_dict['FP']:,}
  • True Negatives (TN):   {metrics_dict['TN']:,}
  • False Negatives (FN):  {metrics_dict['FN']:,}

Error Rates:
  • False Positive Rate:   {metrics_dict['FPR']:.4f}
  • False Negative Rate:   {metrics_dict['FNR']:.4f}
"""
    
    if 'ROC-AUC' in metrics_dict and metrics_dict['ROC-AUC'] is not None:
        summary += f"\nAUC Metrics:\n"
        summary += f"  • ROC-AUC:     {metrics_dict['ROC-AUC']:.4f}\n"
        summary += f"  • PR-AUC:      {metrics_dict['PR-AUC']:.4f}\n"
    
    summary += f"\n{'='*60}\n"
    
    return summary

print("✓ Export helpers defined")


DEFINING DATA EXPORT HELPERS
✓ Export helpers defined


In [8]:
# ============================================================================
# CELL 8: Real-time Prediction Simulation
# ============================================================================
print("\n" + "="*80)
print("DEFINING REAL-TIME PREDICTION SIMULATOR")
print("="*80)

class RealTimeFraudDetector:
    """
    Simulate real-time fraud detection with sliding window
    """
    def __init__(self, model, threshold, timestep, device='cuda'):
        self.model = model
        self.threshold = threshold
        self.timestep = timestep
        self.device = device
        self.buffer = []
        self.predictions_history = []
        self.errors_history = []
        
    def add_transaction(self, transaction):
        """
        Add new transaction to buffer
        
        Args:
            transaction: numpy array of features for one transaction
        """
        self.buffer.append(transaction)
        
        # Keep only last 'timestep' transactions
        if len(self.buffer) > self.timestep:
            self.buffer.pop(0)
        
        # Make prediction if buffer is full
        if len(self.buffer) == self.timestep:
            return self.predict_current()
        else:
            return None, None
    
    def predict_current(self):
        """
        Predict on current buffer
        """
        if len(self.buffer) < self.timestep:
            return None, None
        
        # Create sequence from buffer
        sequence = np.array(self.buffer).reshape(1, self.timestep, -1)
        
        # Predict
        with torch.no_grad():
            inputs = torch.FloatTensor(sequence).to(self.device)
            outputs = self.model(inputs)
            error = torch.mean((inputs - outputs) ** 2).item()
        
        prediction = 1 if error > self.threshold else 0
        
        self.predictions_history.append(prediction)
        self.errors_history.append(error)
        
        return prediction, error
    
    def reset(self):
        """Reset the detector"""
        self.buffer = []
        self.predictions_history = []
        self.errors_history = []

def simulate_realtime_detection(model, X_data, y_data, threshold, timestep, device='cuda'):
    """
    Simulate real-time detection on a dataset
    """
    detector = RealTimeFraudDetector(model, threshold, timestep, device)
    
    predictions = []
    errors = []
    
    # Flatten sequences back to individual transactions
    n_sequences = len(X_data)
    
    print(f"Simulating real-time detection on {n_sequences} sequences...")
    
    for i in range(n_sequences):
        sequence = X_data[i]
        
        # Process each timestep in the sequence
        for t in range(timestep):
            transaction = sequence[t]
            pred, error = detector.add_transaction(transaction)
            
            if pred is not None:
                predictions.append(pred)
                errors.append(error)
        
        if (i + 1) % 1000 == 0:
            print(f"Processed {i+1}/{n_sequences} sequences...")
    
    return np.array(predictions), np.array(errors)

print("✓ Real-time prediction simulator defined")




DEFINING REAL-TIME PREDICTION SIMULATOR
✓ Real-time prediction simulator defined


In [9]:
# ============================================================================
# CELL 9: Statistical Analysis Helpers
# ============================================================================
print("\n" + "="*80)
print("DEFINING STATISTICAL ANALYSIS HELPERS")
print("="*80)

def perform_statistical_test(errors1, errors2, test='mannwhitneyu'):
    """
    Perform statistical test to compare two error distributions
    """
    from scipy import stats
    
    if test == 'mannwhitneyu':
        statistic, pvalue = stats.mannwhitneyu(errors1, errors2, alternative='two-sided')
        test_name = "Mann-Whitney U Test"
    elif test == 'ttest':
        statistic, pvalue = stats.ttest_ind(errors1, errors2)
        test_name = "Independent T-Test"
    elif test == 'ks':
        statistic, pvalue = stats.ks_2samp(errors1, errors2)
        test_name = "Kolmogorov-Smirnov Test"
    else:
        raise ValueError(f"Unknown test: {test}")
    
    result = {
        'test': test_name,
        'statistic': statistic,
        'pvalue': pvalue,
        'significant': pvalue < 0.05
    }
    
    print(f"\n{test_name}:")
    print(f"  Statistic: {statistic:.4f}")
    print(f"  P-value: {pvalue:.6f}")
    print(f"  Significant (α=0.05): {'Yes' if pvalue < 0.05 else 'No'}")
    
    return result

def calculate_error_statistics(errors, labels):
    """
    Calculate detailed statistics for errors by class
    """
    normal_errors = errors[labels == 0]
    fraud_errors = errors[labels == 1]
    
    stats_dict = {
        'Normal': {
            'Mean': np.mean(normal_errors),
            'Median': np.median(normal_errors),
            'Std': np.std(normal_errors),
            'Min': np.min(normal_errors),
            'Max': np.max(normal_errors),
            'Q1': np.percentile(normal_errors, 25),
            'Q3': np.percentile(normal_errors, 75),
            'IQR': np.percentile(normal_errors, 75) - np.percentile(normal_errors, 25)
        },
        'Fraud': {
            'Mean': np.mean(fraud_errors),
            'Median': np.median(fraud_errors),
            'Std': np.std(fraud_errors),
            'Min': np.min(fraud_errors),
            'Max': np.max(fraud_errors),
            'Q1': np.percentile(fraud_errors, 25),
            'Q3': np.percentile(fraud_errors, 75),
            'IQR': np.percentile(fraud_errors, 75) - np.percentile(fraud_errors, 25)
        }
    }
    
    # Convert to DataFrame for nice display
    stats_df = pd.DataFrame(stats_dict).T
    
    print("\nError Statistics by Class:")
    print("="*60)
    print(stats_df.to_string())
    
    return stats_df

print("✓ Statistical analysis helpers defined")


DEFINING STATISTICAL ANALYSIS HELPERS
✓ Statistical analysis helpers defined


In [10]:
# ============================================================================
# CELL 10: Example Usage
# ============================================================================
print("\n" + "="*80)
print("EXAMPLE USAGE")
print("="*80)

print("""
# Example 1: Load model and make predictions
# --------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, checkpoint = load_lstm_model(
    'outputs/models/lstm_ae_fraud_best.pth',
    n_features=35,  # Your number of features
    timestep=10,
    device=device
)
threshold = np.load('outputs/models/threshold_fraud.npy')

# Make predictions
X_test = np.load('outputs/data/X_test_fraud.npy')
y_test = np.load('outputs/data/y_test_fraud.npy')

predictions, errors, reconstructions = predict_anomalies(
    model, X_test, threshold, device
)

# Calculate metrics
metrics = calculate_all_metrics(y_test, predictions, errors)
print(create_results_summary(metrics, 'Fraud Dataset'))


# Example 2: Optimize threshold
# ------------------------------
X_val = np.load('outputs/data/X_val_fraud.npy')
y_val = np.load('outputs/data/y_val_fraud.npy')

_, val_errors, _ = predict_anomalies(model, X_val, 0, device)

optimal_threshold, best_f1, results_df, fig = find_optimal_threshold(
    val_errors, y_val, metric='f1'
)
plt.show()


# Example 3: Visualize reconstruction
# ------------------------------------
fig = plot_sequence_reconstruction(
    X_test, reconstructions, idx=0
)
plt.show()


# Example 4: Compare models
# --------------------------
results = {
    'LSTM-AE': metrics_lstm,
    'Standard AE': metrics_ae,
    'Isolation Forest': metrics_if
}
fig = compare_models(results, metric='F1-Score')
plt.show()


# Example 5: Real-time simulation
# --------------------------------
rt_predictions, rt_errors = simulate_realtime_detection(
    model, X_test, y_test, threshold, timestep=10, device=device
)


# Example 6: Statistical testing
# -------------------------------
normal_errors = errors[y_test == 0]
fraud_errors = errors[y_test == 1]

result = perform_statistical_test(normal_errors, fraud_errors, test='mannwhitneyu')

stats_df = calculate_error_statistics(errors, y_test)


# Example 7: Export to LaTeX
# ---------------------------
results_df = pd.DataFrame({
    'Model': ['LSTM-AE', 'Standard AE', 'Isolation Forest'],
    'Precision': [0.92, 0.85, 0.78],
    'Recall': [0.88, 0.79, 0.72],
    'F1-Score': [0.90, 0.82, 0.75]
})

export_results_to_latex(
    results_df,
    filename='outputs/results_table.tex',
    caption='Comparison of Fraud Detection Models',
    label='tab:model_comparison'
)
""")

print("\n" + "="*80)
print("HELPER FUNCTIONS READY!")
print("="*80)
print("\nAll helper functions are defined and ready to use.")
print("Refer to the example usage above for common scenarios.")
print("\nKey functions available:")
print("  • load_lstm_model() - Load trained models")
print("  • predict_anomalies() - Make predictions")
print("  • calculate_all_metrics() - Compute evaluation metrics")
print("  • find_optimal_threshold() - Optimize detection threshold")
print("  • plot_sequence_reconstruction() - Visualize reconstructions")
print("  • plot_error_analysis() - Analyze error distributions")
print("  • compare_models() - Compare multiple models")
print("  • RealTimeFraudDetector - Simulate real-time detection")
print("  • perform_statistical_test() - Statistical significance testing")
print("  • export_results_to_latex() - Export tables to LaTeX")


EXAMPLE USAGE

# Example 1: Load model and make predictions
# --------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, checkpoint = load_lstm_model(
    'outputs/models/lstm_ae_fraud_best.pth',
    n_features=35,  # Your number of features
    timestep=10,
    device=device
)
threshold = np.load('outputs/models/threshold_fraud.npy')

# Make predictions
X_test = np.load('outputs/data/X_test_fraud.npy')
y_test = np.load('outputs/data/y_test_fraud.npy')

predictions, errors, reconstructions = predict_anomalies(
    model, X_test, threshold, device
)

# Calculate metrics
metrics = calculate_all_metrics(y_test, predictions, errors)
print(create_results_summary(metrics, 'Fraud Dataset'))


# Example 2: Optimize threshold
# ------------------------------
X_val = np.load('outputs/data/X_val_fraud.npy')
y_val = np.load('outputs/data/y_val_fraud.npy')

_, val_errors, _ = predict_anomalies(model, X_val, 0, device)

optimal_threshol